# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanNawaz14/FlyRank-ML-Internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


### Notebook Overview: Working with Large Datasets Efficiently

This notebook is designed to teach you how to handle very large datasets, specifically a ~79 million-row marketing performance dataset hosted on Hugging Face, without needing to download the entire dataset into your local machine's memory. It leverages **DuckDB**, an in-process SQL OLAP database, to perform heavy data processing directly on the remote Parquet files, and then brings only the aggregated results into pandas for further analysis and machine learning.

**The main goals of this notebook are to:**
1.  **Connect to a remote dataset:** Establish a connection to the Hugging Face dataset using DuckDB.
2.  **Efficiently build features:** Demonstrate how to use SQL within DuckDB to aggregate and transform raw data into features, avoiding memory issues by only pulling small results into pandas.
3.  **Train a machine learning model:** Use the engineered features to train a simple classification model to predict content decline.

Let's go through each part of the notebook in more detail.

In [ ]:
%pip -q install duckdb huggingface_hub


### Setup and Hugging Face Token

This first section handles the necessary setup:

*   **`%pip -q install duckdb huggingface_hub`**: Installs the required Python libraries: `duckdb` for in-process SQL querying and `huggingface_hub` for interacting with Hugging Face datasets.
*   **Hugging Face Token**: The code then securely retrieves your Hugging Face read token. This token is crucial for authenticating your access to the private `FlyRank/internship-warehouse` dataset hosted on Hugging Face. It prioritizes fetching the token from environment variables or Colab secrets to prevent it from being exposed in the notebook and to avoid interrupting execution with a manual prompt if the notebook reconnects.

In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


### 1. Connect DuckDB to the Release

This section establishes the connection between DuckDB and the Hugging Face dataset.

*   **`import duckdb`**: Imports the DuckDB library.
*   **`con = duckdb.connect()`**: Creates an in-memory DuckDB database connection.
*   **`CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')`**: This is a key step. It tells DuckDB how to authenticate with Hugging Face using your provided token. Once this secret is registered, DuckDB can seamlessly read Parquet files directly from the Hugging Face dataset as if they were local.
*   **`REL = 'hf://datasets/FlyRank/internship-warehouse'`**: Defines the base path to the Hugging Face dataset.
*   **`TABLES` dictionary**: This dictionary maps logical table names (e.g., `dim_clients`, `fact_daily`) to their actual physical paths within the Hugging Face dataset. Notice how `fact_daily` uses a wildcard (`**/*.parquet`) to read across multiple partitioned Parquet files.
*   **Counting rows**: The loop iterates through the defined tables and uses `con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]` to count the rows. This demonstrates a crucial advantage: DuckDB can get these counts by reading **only the Parquet metadata**, not the entire dataset, which is why it completes very quickly even for the 79 million-row `fact_daily` table. This confirms that DuckDB is working efficiently by only touching what's necessary.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


### 2. Know your panel before you model it

This section emphasizes understanding the characteristics of your data before building models, particularly when dealing with panel data (data collected over time for multiple entities).

*   **`clients = con.sql(...).df()`**: This SQL query fetches client-specific metadata from the `dim_clients` table. It retrieves `client_hash_id`, `access_profile`, and the start dates for GSC (Google Search Console) and GA4 (Google Analytics 4) data.
*   **Unbalanced panel**: The markdown explains that the `gsc_data_start` and `ga4_data_start` columns show that the history depth (how long data has been collected) varies for different clients. This is called an "unbalanced panel." It's important to be aware of this for feature engineering and model training, as some clients might have more historical data than others.
*   **`print('clients with 12+ months of GSC history:', ...)`**: This line calculates and prints how many clients have at least 12 months of GSC data, giving an idea of data availability.

In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [ ]:
clients.shape

(104, 4)

## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


### 3. Build features with SQL, not with RAM

This is a core demonstration of the notebook's efficiency principle: **perform complex aggregations directly in DuckDB using SQL to build features, and then only load the much smaller, aggregated feature table into pandas.**

*   **SQL Query (`features = con.sql(...).df()`):**
    *   **`WITH bounds AS (...)`**: Defines a common table expression (CTE) to find the maximum `report_date` in the `fact_daily` table. This acts as the anchor for defining time windows.
    *   **`windowed AS (...)`**: This is where the feature engineering happens. It groups the `fact_daily` data by `client_hash_id` and `content_hash_id`.
    *   It calculates several

momentum features

:
        *   `imp_last30`: Sum of GSC impressions in the last 30 days.
        *   `imp_prev30`: Sum of GSC impressions in the 30 days *before* the last 30 days (i.e., days 31-60 prior to `end_d`).
        *   `clk_last30`: Sum of GSC clicks in the last 30 days.
        *   `pos_last30`: Average GSC position in the last 30 days.
    *   The `WHERE f.report_date > b.end_d - INTERVAL 60 DAY` clause ensures that only the relevant 60 days of data are considered for each content item.
    *   `HAVING imp_prev30 >= 100`: This filter ensures that only content items with a minimum impression count in the `prev30` window are included, focusing on items with sufficient historical activity.
*   **`.df()`**: Converts the DuckDB query result directly into a pandas DataFrame.
*   **`print(f'{len(features):,} content items with enough history')`**: Shows the number of content items that met the criteria after the aggregation. This DataFrame is significantly smaller than the original `fact_daily` table, making it manageable in pandas.

This section is highlighted as the

heaviest cell

because it involves reading data from the remote Parquet files over the network, but critically, your local RAM usage remains low as DuckDB handles the filtering and aggregation before pandas sees the data.

In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


### 4. Add query-level signals

This section enriches the `features` DataFrame by adding more granular query-level information, again using DuckDB for efficient data retrieval.

*   **`qsignals = con.sql(...).df()`**: This SQL query processes the `fact_query_90d` table, which contains data about how different queries contribute to a page's impressions.
    *   It groups by `content_hash_id` and calculates various metrics:
        *   `visible_queries`: The number of distinct queries for which the content was visible.
        *   `rare_share`, `anon_share`: The proportion of impressions coming from rare or anonymized queries.
        *   `top_query_impressions`: Impressions from the single top-performing query.
        *   `kept_impressions`: Total impressions for the content item within the 90-day window.
    *   `ANY_VALUE` is used for columns like `content_visible_query_count` where the value is consistent across the grouped rows for a given `content_hash_id`.
*   **`qsignals['top_query_share'] = ...`**: Calculates the share of impressions accounted for by the single top query, providing insight into query concentration.
*   **`data = features.merge(qsignals, on='content_hash_id', how='left')`**: Merges these new query-level signals into the existing `features` DataFrame. The `left` merge ensures that all content items from `features` are kept.

In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


### 5. A first honest model

This final section uses the engineered features to build and evaluate a baseline machine learning model. The emphasis here is on building an "honest" model, meaning one that avoids data leakage.

*   **`data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)`**: This line defines the **label** for the classification task: `is_declining`. A content item is labeled as declining (1) if its impressions in the last 30 days (`imp_last30`) are less than 80% of its impressions in the previous 30 days (`imp_prev30`). This label is carefully constructed to avoid leakage with the features.
*   **`feature_cols`**: Specifies the features to be used in the model. Notice that `imp_last30` is NOT used as a feature, as it's directly used to define the label, which would cause leakage.
*   **`model_data = data.dropna(subset=feature_cols)`**: Removes rows where any of the selected feature columns have missing values.
*   **`X, y = model_data[feature_cols], model_data['is_declining']`**: Splits the data into features (`X`) and the target variable (`y`).
*   **`train_test_split`**: Splits the data into training and testing sets. `stratify=y` ensures that the proportion of declining/non-declining content items is similar in both the training and test sets.
*   **`RandomForestClassifier`**: A Random Forest classifier is initialized and trained on the training data (`X_tr`, `y_tr`). `n_jobs=-1` uses all available CPU cores for faster training.
*   **Evaluation**: The code then prints:
    *   `base rate`: The accuracy you would get by simply predicting the majority class (e.g., if 63% of items are not declining, always predicting 'not declining' would give 63% accuracy). This is a crucial baseline.
    *   `classification_report`: A detailed report showing precision, recall, and F1-score for each class (declining vs. not declining), as well as overall accuracy. This allows for a more nuanced understanding of the model's performance beyond just accuracy, especially important for imbalanced classes.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.551     0.337     0.418      9389
           1      0.686     0.840     0.755     16162

    accuracy                          0.655     25551
   macro avg      0.618     0.589     0.587     25551
weighted avg      0.636     0.655     0.631     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


### Your Turn Task 1 & 2: 90-Day Window and New Feature

I will now re-run the feature engineering step (similar to Section 3) with two changes:
1.  **90-day window:** The `fact_daily` data considered for calculating `imp_last30` and `imp_prev30` will span a total of 90 days prior to the `end_d`.
2.  **New feature - Position Volatility (`pos_std_last30`):** I will add a new feature that calculates the standard deviation of `gsc_avg_position` within the last 30 days. This can capture how stable or volatile a content item's ranking is, which might be a signal for decline or improvement. I'll also adjust the `HAVING` threshold slightly to `imp_prev30 >= 50` to potentially include more content items.

In [ ]:
import numpy as np

# Re-run feature engineering with 90-day window and new feature
features_v2 = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30,
               STDDEV_SAMP(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS pos_std_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY -- Changed from 60 to 90 days
        GROUP BY 1, 2
        HAVING imp_prev30 >= 50 -- Adjusted threshold
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features_v2):,} content items with enough history (90-day window, prev30 >= 50)')
display(features_v2.head())

# Re-merge with query-level signals
data_v2 = features_v2.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data_v2):,} rows')
display(data_v2.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

155,903 content items with enough history (90-day window, prev30 >= 50)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,pos_std_last30
0,client_3ffa76342f366962,content_b89167cd03d6ffc1,12.0,102.0,1.0,5.104167,2.467145
1,client_3ffa76342f366962,content_f33fd2cd160185e7,41.0,130.0,0.0,6.323529,2.326046
2,client_e547b89c05043229,content_2e296120acb03e93,2346.0,5207.0,0.0,38.156281,6.544167
3,client_e547b89c05043229,content_516b7c0e8eec0cef,371.0,2494.0,0.0,48.364954,13.492583
4,client_e547b89c05043229,content_38b6c1a9aa29f801,5746.0,12657.0,2.0,40.082120,5.952548


joined: 155,903 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,pos_std_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_3ffa76342f366962,content_b89167cd03d6ffc1,12.0,102.0,1.0,5.104167,2.467145,NaN,NaN,NaN,NaN,NaN,NaN
1,client_3ffa76342f366962,content_f33fd2cd160185e7,41.0,130.0,0.0,6.323529,2.326046,NaN,NaN,NaN,NaN,NaN,NaN
2,client_e547b89c05043229,content_2e296120acb03e93,2346.0,5207.0,0.0,38.156281,6.544167,43.0,0.042632,0.545876,409.0,3108.0,0.131596
3,client_e547b89c05043229,content_516b7c0e8eec0cef,371.0,2494.0,0.0,48.364954,13.492583,13.0,0.112391,0.679232,305.0,597.0,0.510888
4,client_e547b89c05043229,content_38b6c1a9aa29f801,5746.0,12657.0,2.0,40.082120,5.952548,129.0,0.069119,0.517198,1398.0,7613.0,0.183633


### Your Turn Task 3: GroupShuffleSplit

To ensure our model generalizes well and avoids data leakage across clients, I will replace the simple random `train_test_split` with `GroupShuffleSplit`. This method ensures that all data points belonging to a particular `client_hash_id` are either entirely in the training set or entirely in the test set. This is crucial for evaluating model performance in a scenario where we might encounter new clients.

In [12]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Define the label for declining content based on the new features_v2
data_v2['is_declining'] = (data_v2['imp_last30'] < 0.8 * data_v2['imp_prev30']).astype(int)

# Update feature columns. Removed 'visible_queries', 'rare_share', 'anon_share',
# and 'top_query_share' due to data leakage with the 'is_declining' label,
# as the source table `fact_content_query_90d` overlaps with the label's time window.
feature_cols_v2 = ['imp_prev30', 'pos_std_last30']

# Drop rows with NaN in feature columns, as pos_std_last30 might introduce some
model_data_v2 = data_v2.dropna(subset=feature_cols_v2)

X_v2, y_v2 = model_data_v2[feature_cols_v2], model_data_v2['is_declining']
groups_v2 = model_data_v2['client_hash_id']

# Initialize GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

# Generate train/test indices using GroupShuffleSplit
for train_idx, test_idx in gss.split(X_v2, y_v2, groups_v2):
    X_tr_gss, X_te_gss = X_v2.iloc[train_idx], X_v2.iloc[test_idx]
    y_tr_gss, y_te_gss = y_v2.iloc[train_idx], y_v2.iloc[test_idx]

print(f'Training set size (GroupShuffleSplit): {len(X_tr_gss):,} rows')
print(f'Test set size (GroupShuffleSplit): {len(X_te_gss):,} rows')

# Train the model with the new split
model_gss = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_gss, y_tr_gss)

# Evaluate the model
print(f'\nBase rate (always predict majority): {max(y_te_gss.mean(), 1 - y_te_gss.mean()):.3f}')
print(classification_report(y_te_gss, model_gss.predict(X_te_gss), digits=3))

Training set size (GroupShuffleSplit): 89,355 rows
Test set size (GroupShuffleSplit): 55,415 rows

Base rate (always predict majority): 0.883
              precision    recall  f1-score   support

           0      0.385     0.183     0.248      6472
           1      0.899     0.961     0.929     48943

    accuracy                          0.870     55415
   macro avg      0.642     0.572     0.588     55415
weighted avg      0.839     0.870     0.850     55415



### Your Turn & Working Locally

This final section provides suggestions for further exploration and improvement, challenging you to build on the concepts learned:

1.  **Re-run with a 90-day window**: Encourages you to modify the SQL query in Section 3 to use a different time window for feature calculation, testing its impact.
2.  **Add a new feature**: Asks you to think of and implement an additional feature (e.g., position volatility, weekend share) that might improve the model.
3.  **GroupShuffleSplit**: This is a very important concept for panel data. It challenges you to replace the simple random train-test split with `GroupShuffleSplit` on `client_hash_id`. This type of split ensures that the same client doesn't appear in both the training and testing sets, which is critical for evaluating whether your model **generalizes to new, unseen clients**, rather than just memorizing patterns within existing clients. This is the difference between a

lucky split

and a robust model.

**Working Locally**: The last part provides guidance on how to download specific parts of the Hugging Face dataset if you need to work locally, again emphasizing only downloading the necessary partitions to avoid massive downloads.